# GCN Example — Node Classification on the Karate Club Graph

A **Graph Convolutional Network (GCN)** (Kipf & Welling, 2017) built from scratch with
TensorFlow / Keras — no graph libraries required.

## What problem are we solving?

Normal neural networks expect grid-like inputs (a vector, an image, a sequence). But lots
of real data is a **graph**: social networks, molecules, citation networks, road maps. A
GCN is a neural network that operates directly on graph structure, letting each node's
prediction depend on its **neighbours**.

We use **Zachary's Karate Club** — a classic 34-person social network recorded in the
1970s. A dispute split the club into two factions, one around the instructor (node 0)
and one around the president (node 33). Our task is **semi-supervised node
classification**: we are told the faction of **only 2 nodes** (those two leaders) and
must predict the faction of all 32 others, using nothing but the friendship edges.

## The GCN layer in one line

$$H' = \sigma\big(\hat{A}\,H\,W\big)$$

Read it right-to-left:

- $H$ — the current node features (one row per node).
- $H W$ — a standard linear transform (learned weights $W$), like any dense layer.
- $\hat{A}\,(HW)$ — multiplying by the (normalized) adjacency matrix **mixes each node's
  features with its neighbours'**. This single matrix multiply *is* the 'graph
  convolution' — it is how information flows along edges.
- $\sigma$ — a nonlinearity (ReLU).

Stacking two such layers means each node ends up aware of its neighbours **and its
neighbours' neighbours** (a 2-hop receptive field). That is exactly how the 2 known
labels can reach the whole graph.

> Tested with `tensorflow==2.1` (matches `requirements.txt`). Runs instantly on CPU.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt

print('TensorFlow', tf.__version__)

## 1. Build the Karate Club graph

A graph is fully described by its **adjacency matrix** $A$: a `34x34` matrix where
`A[i, j] = 1` if persons `i` and `j` are friends, else `0`. We hard-code the 78
friendship edges (so there is no dependency on `networkx`) and build a **symmetric**
matrix — friendship goes both ways, so we set both `A[i, j]` and `A[j, i]`.

In [ ]:
edges = [
    (0,1),(0,2),(0,3),(0,4),(0,5),(0,6),(0,7),(0,8),(0,10),(0,11),(0,12),
    (0,13),(0,17),(0,19),(0,21),(0,31),(1,2),(1,3),(1,7),(1,13),(1,17),(1,19),
    (1,21),(1,30),(2,3),(2,7),(2,8),(2,9),(2,13),(2,27),(2,28),(2,32),(3,7),
    (3,12),(3,13),(4,6),(4,10),(5,6),(5,10),(5,16),(6,16),(8,30),(8,32),(8,33),
    (9,33),(13,33),(14,32),(14,33),(15,32),(15,33),(18,32),(18,33),(19,33),
    (20,32),(20,33),(22,32),(22,33),(23,25),(23,27),(23,29),(23,32),(23,33),
    (24,25),(24,27),(24,31),(25,31),(26,29),(26,33),(27,33),(28,31),(28,33),
    (29,32),(29,33),(30,32),(30,33),(31,32),(31,33),(32,33),
]

N = 34  # number of nodes

# Symmetric adjacency matrix
A = np.zeros((N, N), dtype='float32')
for i, j in edges:
    A[i, j] = 1.0
    A[j, i] = 1.0

print('Nodes:', N, '| Edges:', len(edges))

## 2. Normalize the adjacency matrix

We don't use the raw $A$ directly. Kipf & Welling apply two fixes that make message
passing well-behaved:

1. **Add self-loops:** $\tilde{A} = A + I$. Without this, multiplying by $A$ would
   replace each node's features with a sum of its *neighbours'* features and **drop the
   node's own**. The self-loop keeps a node's own information in the mix.
2. **Symmetric normalization:** $\hat{A} = \tilde{D}^{-1/2}\,\tilde{A}\,\tilde{D}^{-1/2}$,
   where $\tilde{D}$ is the diagonal **degree** matrix of $\tilde{A}$. This rescales by
   node degree so that high-degree 'hub' nodes don't blow up the feature magnitudes —
   roughly, each neighbour contributes a weighted average instead of a raw sum.

The result $\hat{A}$ is computed once and reused at every layer; it is a fixed property
of the graph, not something we learn.

In [ ]:
A_tilde = A + np.eye(N, dtype='float32')        # add self-loops
D = np.sum(A_tilde, axis=1)                      # degree of each node
D_inv_sqrt = np.diag(1.0 / np.sqrt(D))
A_hat = (D_inv_sqrt @ A_tilde @ D_inv_sqrt).astype('float32')
A_hat = tf.constant(A_hat)

print('Normalized adjacency shape:', A_hat.shape)

## 3. Features and labels

**Features.** This dataset has no node attributes, so we use a one-hot **identity
matrix** as input features (`X = I`). This is a standard trick: it gives every node a
unique, learnable 'slot', so the first GCN layer effectively learns one embedding vector
per node. (If you *did* have real attributes — age, interests, etc. — you would put them
in `X` instead.)

**Labels.** The whole point of *semi-supervised* learning is that labels are scarce. We
reveal only the two faction leaders — node `0` (instructor, class 0) and node `33`
(president, class 1). The remaining 32 labels (`y_true`) are kept aside purely to *score*
the model at the end; the network never trains on them.

In [ ]:
X = tf.eye(N)  # identity features, shape (34, 34)

# Ground-truth factions (used only for final evaluation / plotting)
y_true = np.array([
    0,0,0,0,0,0,0,0,1,1,0,0,0,0,1,1,0,0,1,0,1,0,1,1,
    1,1,1,1,1,1,1,1,1,1
])

# Semi-supervised: we only 'know' two labels
labeled_nodes = [0, 33]
y_labeled = tf.constant([y_true[0], y_true[33]])

print('Labeled nodes:', labeled_nodes, '-> labels', y_labeled.numpy())

## 4. Define a GCN layer

A custom Keras layer implementing $\hat{A}\,(X W)$. The only **learnable** parameter is
the weight matrix `w` (created in `build`); the normalized adjacency `A_hat` is a fixed
constant captured at construction. In `call` we first apply the linear map `X W`, then
multiply by `A_hat` to mix neighbours, then apply the activation.

In [ ]:
class GCNLayer(layers.Layer):
    def __init__(self, units, A_hat, activation=None, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.A_hat = A_hat
        self.activation = tf.keras.activations.get(activation)

    def build(self, input_shape):
        self.w = self.add_weight(
            shape=(input_shape[-1], self.units),
            initializer='glorot_uniform',
            trainable=True,
            name='w',
        )

    def call(self, x):
        x = tf.matmul(x, self.w)         # X W   (linear transform)
        x = tf.matmul(self.A_hat, x)     # A_hat (X W)  -> mix neighbours
        return self.activation(x) if self.activation else x

## 5. Build the 2-layer GCN

Why exactly two layers? Each GCN layer spreads information **one hop** along the edges.
With two layers, the signal from the two labelled leaders reaches everything within two
hops — which, on this small, well-connected graph, is essentially the whole club.
(Going much deeper tends to *over-smooth*: all node features blur toward the same value.)

- **Layer 1**: `34 → 16` features, ReLU. Produces a 16-dim hidden embedding per node; we
  stash it in `self.embeddings` so we can visualise it later.
- **Layer 2**: `16 → 2` features, no activation — raw class logits.

We use sparse-categorical cross-entropy on logits and an Adam optimizer with a fairly
high learning rate (`0.05`), which is fine because the 'dataset' is tiny.

In [ ]:
class GCN(tf.keras.Model):
    def __init__(self, A_hat, hidden=16, classes=2):
        super().__init__()
        self.gc1 = GCNLayer(hidden, A_hat, activation='relu')
        self.gc2 = GCNLayer(classes, A_hat, activation=None)

    def call(self, x):
        x = self.gc1(x)
        self.embeddings = x          # keep hidden embeddings for plotting
        return self.gc2(x)

model = GCN(A_hat)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
optimizer = tf.keras.optimizers.Adam(0.05)

## 6. Train

Here is the elegant part of semi-supervised GCNs. On every step we run the **whole
graph** through the model in one shot (all 34 nodes), but we compute the loss on **only
the 2 labelled nodes** — we `tf.gather` their rows out of the output.

Even though the loss sees just 2 nodes, the gradient flows backward through the two
`A_hat` multiplications, so it touches the weights that shaped *every* node's
representation. In effect, the labels of the two leaders propagate outward along the
friendship edges. We print the accuracy on the full graph every 20 epochs to watch it
climb.

In [ ]:
EPOCHS = 200

for epoch in range(1, EPOCHS + 1):
    with tf.GradientTape() as tape:
        logits = model(X)
        logits_labeled = tf.gather(logits, labeled_nodes)
        loss = loss_fn(y_labeled, logits_labeled)

    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))

    if epoch % 20 == 0 or epoch == 1:
        preds = tf.argmax(model(X), axis=1).numpy()
        acc = (preds == y_true).mean()
        print(f'Epoch {epoch:3d}/{EPOCHS}  loss={float(loss):.4f}  full-graph acc={acc:.3f}')

## 7. Results

From just two labelled nodes the GCN classifies the rest of the club. Let's compare its
predictions against the true factions. You should typically see **30+/34 correct** — not
bad for a model that was told only 2 answers!

In [ ]:
preds = tf.argmax(model(X), axis=1).numpy()
acc = (preds == y_true).mean()
print(f'Final accuracy on all {N} nodes: {acc:.1%}')
print('Predicted factions:', preds)
print('True factions:     ', y_true)

## 8. Visualize the learned embeddings

The 16-dim hidden embeddings (the output of layer 1) live in a space the GCN organised so
the two factions are linearly separable. We project them down to 2D with **PCA**
(implemented here with a plain SVD, so there is no `sklearn` dependency) and plot each
node coloured by its predicted faction. The two labelled leaders are circled in black.
You should see two well-separated clusters.

In [ ]:
_ = model(X)  # populate model.embeddings
emb = model.embeddings.numpy()

# Simple 2D PCA via SVD (no sklearn dependency)
emb_centered = emb - emb.mean(axis=0)
u, s, vt = np.linalg.svd(emb_centered, full_matrices=False)
coords = emb_centered @ vt[:2].T

plt.figure(figsize=(7, 6))
plt.scatter(coords[:, 0], coords[:, 1], c=preds, cmap='coolwarm', s=200, edgecolors='gray')
for n in labeled_nodes:
    plt.scatter(coords[n, 0], coords[n, 1], s=320, facecolors='none', edgecolors='black', linewidths=2.5)
for i, (x_, y_) in enumerate(coords):
    plt.text(x_, y_, str(i), ha='center', va='center', fontsize=8)
plt.title('GCN node embeddings (PCA) — colored by predicted faction')
plt.xlabel('PC 1'); plt.ylabel('PC 2')
plt.show()

### Things to try

- Change `hidden` units or `EPOCHS` and see how the separation changes.
- Label different/more nodes and observe the effect on accuracy — what happens if you
  only label node 0?
- Replace the identity features with real node attributes if you have them.
- Add a third GCN layer — does deeper help, or does *over-smoothing* hurt on such a small
  graph?
- Remove the self-loop (`A_tilde = A`) and watch accuracy drop — a concrete demo of why
  that `+ I` matters.